# WaWa 點點樂 — 單詞擷取（2026 更新版）

從原住民族委員會「族語 e 樂園」WaWa 點點樂線上版擷取 **6 大主題、60 個核心單詞**，涵蓋 **16 族 42 方言**。

> **2026 更新說明**：族語 e 樂園於 2024/12 改版，舊版以 Selenium 點選 `switcher` 切換方言的方式已失效。
> 新版改用官網的偏好方言機制 —— 先 `POST member/set_prefer_dialect.php` 設定方言（記在 session），
> 再 `GET wawa/word.php?pid=N` 即可取得該方言單詞。**不再需要 Selenium / chromedriver。**

資料來源：<https://web.klokah.tw/>（原住民族委員會 · 族語數位中心），僅供族語學習與研究之非商業用途。


## 1. 設定與方言對照表

In [ ]:
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE = "https://web.klokah.tw"
SET_DIALECT = f"{BASE}/member/set_prefer_dialect.php"
WORD_URL = f"{BASE}/wawa/word.php?pid={{pid}}"
HEADERS = {"User-Agent": "Mozilla/5.0"}

# (did, lid, 方言, 族) — 取自官網 dialector.js
DIALECTS = [
    ("1","1","南勢阿美語","阿美語"),("2","1","秀姑巒阿美語","阿美語"),("3","1","海岸阿美語","阿美語"),
    ("4","1","馬蘭阿美語","阿美語"),("5","1","恆春阿美語","阿美語"),
    ("6","2","賽考利克泰雅語","泰雅語"),("7","2","澤敖利泰雅語","泰雅語"),("8","2","汶水泰雅語","泰雅語"),
    ("9","2","萬大泰雅語","泰雅語"),("10","2","四季泰雅語","泰雅語"),("11","2","宜蘭澤敖利泰雅語","泰雅語"),
    ("13","3","賽夏語","賽夏語"),("14","4","邵語","邵語"),
    ("15","5","都達賽德克語","賽德克語"),("16","5","德固達雅賽德克語","賽德克語"),("17","5","德鹿谷賽德克語","賽德克語"),
    ("18","6","卓群布農語","布農語"),("19","6","卡群布農語","布農語"),("20","6","丹群布農語","布農語"),
    ("21","6","巒群布農語","布農語"),("22","6","郡群布農語","布農語"),
    ("23","7","東排灣語","排灣語"),("24","7","北排灣語","排灣語"),("25","7","中排灣語","排灣語"),("26","7","南排灣語","排灣語"),
    ("27","8","東魯凱語","魯凱語"),("28","8","霧台魯凱語","魯凱語"),("29","8","大武魯凱語","魯凱語"),
    ("30","8","多納魯凱語","魯凱語"),("31","8","茂林魯凱語","魯凱語"),("32","8","萬山魯凱語","魯凱語"),
    ("33","9","太魯閣語","太魯閣語"),("34","10","噶瑪蘭語","噶瑪蘭語"),("35","11","鄒語","鄒語"),
    ("38","12","南王卑南語","卑南語"),("39","12","知本卑南語","卑南語"),("40","12","西群卑南語","卑南語"),("41","12","建和卑南語","卑南語"),
    ("42","13","雅美語","雅美語"),("43","14","撒奇萊雅語","撒奇萊雅語"),
    ("36","15","卡那卡那富語","卡那卡那富語"),("37","16","拉阿魯哇語","拉阿魯哇語"),
]

# WaWa 點點樂 6 大主題單元
THEMES = {1:"我長大了", 2:"我的家庭真可愛", 3:"歡樂慶豐收", 4:"部落真好玩", 5:"123大風吹", 6:"山豬飛鼠來點名"}
print(f"共 {len(DIALECTS)} 個方言、{len(THEMES)} 個主題")

## 2. 擷取函式

In [ ]:
def fetch_words(session, pid):
    """抓取目前偏好方言下，某主題 pid 的所有單詞。"""
    html = session.get(WORD_URL.format(pid=pid), headers=HEADERS, timeout=30).text
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for w in soup.select(".showword"):
        ab = w.select_one(".AB")
        ch = w.select_one(".CH")
        audio = w.select_one("a.sm2_button")
        rows.append({
            "族語": ab.get_text(strip=True) if ab else "",
            "中文": ch.get_text(strip=True) if ch else "",
            "音檔": (BASE + "/wawa/" + audio["href"]) if (audio and audio.has_attr("href")) else "",
        })
    return rows


def scrape(dialects=DIALECTS, pids=range(1, 7), pause=0.3):
    """逐方言設定偏好後，擷取 6 大主題單詞，回傳 DataFrame。"""
    records = []
    for did, lid, dch, lch in dialects:
        session = requests.Session()
        session.get(WORD_URL.format(pid=1), headers=HEADERS, timeout=30)        # 取得 session
        session.post(SET_DIALECT, data={"did": did}, headers=HEADERS, timeout=30)  # 設定偏好方言
        for pid in pids:
            for r in fetch_words(session, pid):
                records.append({"族": lch, "方言": dch, "did": did, "主題": THEMES.get(pid, pid), **r})
            time.sleep(pause)
        print(f"  OK {lch} / {dch} (did={did})  累計 {len(records)} 筆")
    return pd.DataFrame(records)

## 3. 小測試（單一方言）

In [ ]:
# 先用南勢阿美語驗證流程正常
demo = scrape([("1","1","南勢阿美語","阿美語")], pids=range(1, 3))
demo.head(10)

## 4. 完整擷取（42 方言 × 6 主題）並輸出 CSV

In [ ]:
df = scrape()  # 全部跑完約需數分鐘
print("總筆數：", df.shape)
df.to_csv("wawa_words.csv", index=False, encoding="utf-8-sig")
print("已輸出 wawa_words.csv")
df.head()